In [49]:
# Shared data loading and plot styling.
import json
import re
from collections import Counter
from datetime import datetime
from dateutil.relativedelta import relativedelta
from itertools import cycle
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

In [50]:
# Plot 1: Interactive filler-word percentages per recording.
import plotly.graph_objects as go

if not records:
    print('No recordings available to plot.')
else:
    # Build views for dropdown options using most recent recordings.
    dropdown_options = [
        ('Last 5 recordings', 5),
        ('Last 10 recordings', 10),
        ('Last 15 recordings', 15),
        ('All recordings', None),
    ]

    def build_view(limit):
        recency_sorted = sorted(records, key=lambda record: (record['recording_dt'], record['recording_id']), reverse=True)
        selected = recency_sorted if limit is None else recency_sorted[:limit]
        selected = sorted(selected, key=lambda record: (record['filler_percentage'], record['recording_dt']), reverse=True)

        labels = [
            f"{record['recording_label']}  •  {record['recording_dt'].strftime('%b %d, %Y')}"
            for record in selected
        ]
        totals = np.array([record['filler_percentage'] for record in selected], dtype=float)

        x_by_word = []
        custom_by_word = []
        visible_by_word = []

        for word in filler_words:
            values = np.array(
                [
                    100.0 * record['filler_counts'].get(word, 0) / max(record['total_words'], 1)
                    for record in selected
                ],
                dtype=float,
            )
            custom = np.column_stack(
                [
                    np.array([word] * len(selected), dtype=object),
                    np.array([record['recording_label'] for record in selected], dtype=object),
                    totals,
                    np.array([record['total_words'] for record in selected], dtype=float),
                ]
            )
            x_by_word.append(values)
            custom_by_word.append(custom)
            visible_by_word.append(bool(np.any(values > 0)))

        return {
            'labels': labels,
            'totals': totals,
            'x_by_word': x_by_word,
            'custom_by_word': custom_by_word,
            'visible_by_word': visible_by_word,
        }

    views = {name: build_view(limit) for name, limit in dropdown_options}
    default_view_name = 'Last 5 recordings'
    default_view = views[default_view_name]

    fig = go.Figure()
    for idx, word in enumerate(filler_words):
        fig.add_trace(
            go.Bar(
                y=default_view['labels'],
                x=default_view['x_by_word'][idx],
                name=word,
                orientation='h',
                marker_color=word_colors[word],
                customdata=default_view['custom_by_word'][idx],
                visible=default_view['visible_by_word'][idx],
                hovertemplate=(
                    '<b>%{customdata[1]}</b><br>'
                    'Filler word: %{customdata[0]}<br>'
                    'Word share: %{x:.2f}%<br>'
                    'Total filler: %{customdata[2]:.2f}%<br>'
                    'Total words: %{customdata[3]:.0f}<extra></extra>'
                ),
            )
        )

    dropdown_buttons = []
    for name, _ in dropdown_options:
        view = views[name]
        dropdown_buttons.append(
            {
                'label': name,
                'method': 'update',
                'args': [
                    {
                        'x': view['x_by_word'],
                        'y': [view['labels']] * len(filler_words),
                        'customdata': view['custom_by_word'],
                        'visible': view['visible_by_word'],
                    },
                    {
                        'title': f'Filler words as percentages per recording ({name})',
                    },
                ],
            }
        )

    fig.update_layout(
        barmode='stack',
        title=f'Filler words as percentages per recording ({default_view_name})',
        xaxis_title='Percentage of all spoken words',
        yaxis_title='Recording',
        xaxis=dict(range=[0, 100], ticksuffix='%'),
        template='plotly_white',
        height=max(560, 90 + 26 * len(default_view['labels'])),
        margin=dict(l=230, r=35, t=80, b=140),
        legend=dict(
            orientation='h',
            yanchor='top',
            y=-0.22,
            xanchor='left',
            x=0,
            title_text='Filler word',
            font=dict(size=11),
        ),
        updatemenus=[
            {
                'buttons': dropdown_buttons,
                'direction': 'down',
                'showactive': True,
                'x': 0.0,
                'xanchor': 'left',
                'y': 1.17,
                'yanchor': 'top',
                'bgcolor': 'white',
                'bordercolor': '#B7C7E5',
                'borderwidth': 1,
            }
        ],
    )
    fig.update_yaxes(autorange='reversed')

    fig.show()

In [51]:
# Plot 2: Filler words over time as interactive line chart with time frame selector.
import plotly.graph_objects as go
from dateutil.relativedelta import relativedelta
from datetime import timedelta

if not records:
    print('No recordings available to plot.')
else:
    time_ordered_records = sorted(records, key=lambda record: (record['recording_dt'], record['recording_id']))
    latest_date = time_ordered_records[-1]['recording_dt'] if time_ordered_records else None
    
    # Calculate Monday and Sunday of this week
    monday_this_week = latest_date - timedelta(days=latest_date.weekday())
    sunday_this_week = monday_this_week + timedelta(days=6)
    
    # Calculate Monday and Sunday of last week
    monday_last_week = monday_this_week - timedelta(days=7)
    sunday_last_week = monday_last_week + timedelta(days=6)
    
    # Define time frame filters with start and end dates
    time_frames = {
        'This week': ('week', monday_this_week, sunday_this_week),
        'Last week': ('week', monday_last_week, sunday_last_week),
        'Month': ('month', latest_date - relativedelta(months=1), latest_date),
        '6 months': ('period', latest_date - relativedelta(months=6), latest_date),
        'Year': ('period', latest_date - relativedelta(years=1), latest_date),
        'All time': ('all', None, None),
    }
    
    def get_filtered_data(frame_type, start_date, end_date):
        """Filter records based on date range"""
        if frame_type == 'all':
            return time_ordered_records
        if start_date is None:
            return time_ordered_records
        return [r for r in time_ordered_records if start_date <= r['recording_dt'] <= end_date]
    
    # Build data for each time frame
    frame_data = {}
    for frame_name, (frame_type, start_date, end_date) in time_frames.items():
        filtered_records = get_filtered_data(frame_type, start_date, end_date)
        
        if not filtered_records:
            # Store empty state
            frame_data[frame_name] = {
                'traces': {},
                'max_value': 5.0,
                'empty': True,
                'message': f"You have not done any recordings in this chosen timeframe: '{frame_name}'",
                'frame_type': frame_type,
            }
        else:
            dates = [r['recording_dt'] for r in filtered_records]
            
            traces_data = {}
            max_value = 0
            
            for word in filler_words:
                values = np.array(
                    [
                        100.0 * r['filler_counts'].get(word, 0) / max(r['total_words'], 1)
                        for r in filtered_records
                    ],
                    dtype=float,
                )
                max_value = max(max_value, np.max(values) if values.size else 0)
                traces_data[word] = {'dates': dates, 'values': values}
            
            frame_data[frame_name] = {
                'traces': traces_data,
                'max_value': max(5.0, max_value * 1.2),
                'empty': False,
                'frame_type': frame_type,
            }
    
    # Create figure with first time frame
    default_frame = 'All time'
    fig = go.Figure()
    
    default_data = frame_data[default_frame]
    
    # Choose hover date format based on frame type
    hover_date_fmt = '%b %d, %Y'
    if default_data['frame_type'] == 'week':
        hover_date_fmt = '%a, %b %d'
    elif default_data['frame_type'] in ['month', 'period']:
        hover_date_fmt = '%b %d, %Y'
    
    if not default_data['empty']:
        for word in filler_words:
            trace_info = default_data['traces'][word]
            fig.add_trace(
                go.Scatter(
                    x=trace_info['dates'],
                    y=trace_info['values'],
                    name=word,
                    mode='lines+markers',
                    line=dict(color=word_colors[word], width=2.5),
                    marker=dict(size=6),
                    hovertemplate=(
                        '<b>%{x|' + hover_date_fmt + '}</b><br>'
                        'Filler word: ' + word + '<br>'
                        'Percentage: %{y:.2f}%<extra></extra>'
                    ),
                )
            )
    else:
        # Add empty annotation
        fig.add_annotation(
            text=default_data['message'],
            xref='paper',
            yref='paper',
            x=0.5,
            y=0.5,
            showarrow=False,
            font=dict(size=16, color='#999999'),
        )
    
    # Create time frame dropdown buttons
    timeframe_buttons = []
    for frame_name, (frame_type, start_date, end_date) in time_frames.items():
        frame_info = frame_data[frame_name]
        
        # Determine hover date format for this frame
        if frame_type == 'week':
            hover_fmt = '%a, %b %d'
        else:
            hover_fmt = '%b %d, %Y'
        
        if frame_info['empty']:
            # Empty state button
            timeframe_buttons.append(
                {
                    'label': frame_name,
                    'method': 'update',
                    'args': [
                        {
                            'x': [[]],
                            'y': [[]],
                        },
                        {
                            'title': f'Filler words over time ({frame_name})',
                            'annotations': [{
                                'text': frame_info['message'],
                                'xref': 'paper',
                                'yref': 'paper',
                                'x': 0.5,
                                'y': 0.5,
                                'showarrow': False,
                                'font': dict(size=16, color='#999999'),
                            }],
                            'yaxis.range': [0, 5.0],
                        },
                    ],
                }
            )
        else:
            # Normal state button
            new_x = [frame_info['traces'][word]['dates'] for word in filler_words]
            new_y = [frame_info['traces'][word]['values'] for word in filler_words]
            
            timeframe_buttons.append(
                {
                    'label': frame_name,
                    'method': 'update',
                    'args': [
                        {
                            'x': new_x,
                            'y': new_y,
                            'hovertemplate': ['<b>%{x|' + hover_fmt + '}</b><br>Filler word: ' + word + '<br>Percentage: %{y:.2f}%<extra></extra>' for word in filler_words],
                        },
                        {
                            'title': f'Filler words over time ({frame_name})',
                            'annotations': [],
                            'yaxis.range': [0, frame_info['max_value']],
                        },
                    ],
                }
            )

    fig.update_layout(
        title=f'Filler words over time ({default_frame})',
        xaxis_title='Recording date',
        yaxis_title='Percentage of spoken words (%)',
        template='plotly_white',
        height=600,
        hovermode='x unified',
        legend=dict(
            orientation='v',
            yanchor='top',
            y=0.99,
            xanchor='left',
            x=0.01,
            font=dict(size=11),
        ),
        yaxis=dict(range=[0, default_data['max_value']]),
        updatemenus=[
            {
                'buttons': timeframe_buttons,
                'direction': 'down',
                'showactive': True,
                'x': 0.0,
                'xanchor': 'left',
                'y': 1.12,
                'yanchor': 'top',
                'bgcolor': 'white',
                'bordercolor': '#B7C7E5',
                'borderwidth': 1,
            }
        ],
    )

    fig.show()

In [52]:
# Plot 3: Interactive pitch range per session over time with time frame selector.
import plotly.graph_objects as go
from dateutil.relativedelta import relativedelta
from datetime import timedelta

if not records:
    print('No recordings available to plot.')
else:
    pitch_records = [record for record in records if np.isfinite(record['pitch_avg_variation'])]
    if not pitch_records:
        print('No voiced pitch values available to plot.')
    else:
        latest_date = pitch_records[-1]['recording_dt'] if pitch_records else None
        
        # Calculate Monday and Sunday of this week
        monday_this_week = latest_date - timedelta(days=latest_date.weekday())
        sunday_this_week = monday_this_week + timedelta(days=6)
        
        # Calculate Monday and Sunday of last week
        monday_last_week = monday_this_week - timedelta(days=7)
        sunday_last_week = monday_last_week + timedelta(days=6)
        
        # Define time frame filters with start and end dates
        time_frames = {
            'This week': ('week', monday_this_week, sunday_this_week),
            'Last week': ('week', monday_last_week, sunday_last_week),
            'Month': ('month', latest_date - relativedelta(months=1), latest_date),
            '6 months': ('period', latest_date - relativedelta(months=6), latest_date),
            'Year': ('period', latest_date - relativedelta(years=1), latest_date),
            'All time': ('all', None, None),
        }
        
        def get_filtered_pitch_data(frame_type, start_date, end_date):
            """Filter records based on date range"""
            if frame_type == 'all':
                return pitch_records
            if start_date is None:
                return pitch_records
            return [r for r in pitch_records if start_date <= r['recording_dt'] <= end_date]
        
        # Build data for each time frame
        frame_data = {}
        target_low, target_high = 3.0, 5.0
        
        for frame_name, (frame_type, start_date, end_date) in time_frames.items():
            filtered_records = get_filtered_pitch_data(frame_type, start_date, end_date)
            
            if not filtered_records:
                # Store empty state
                frame_data[frame_name] = {
                    'empty': True,
                    'message': f"You have not done any recordings in this chosen timeframe: '{frame_name}'",
                    'upper_limit': target_high + 2.0,
                    'frame_type': frame_type,
                }
            else:
                dates = [r['recording_dt'] for r in filtered_records]
                avg_variation = np.array([r['pitch_avg_variation'] for r in filtered_records], dtype=float)
                min_variation = np.array([r['pitch_min_variation'] for r in filtered_records], dtype=float)
                max_variation = np.array([r['pitch_max_variation'] for r in filtered_records], dtype=float)
                overall_average = float(np.nanmean(avg_variation))
                upper_limit = max(target_high + 2.0, float(np.nanmax(max_variation)) * 1.15 if max_variation.size else target_high + 2.0)
                
                frame_data[frame_name] = {
                    'dates': dates,
                    'avg_variation': avg_variation,
                    'min_variation': min_variation,
                    'max_variation': max_variation,
                    'overall_average': overall_average,
                    'upper_limit': upper_limit,
                    'empty': False,
                    'frame_type': frame_type,
                }
        
        # Create figure with first time frame
        default_frame = 'All time'
        default_data = frame_data[default_frame]
        
        # Choose hover date format based on frame type
        hover_date_fmt = '%b %d, %Y'
        if default_data['frame_type'] == 'week':
            hover_date_fmt = '%a, %b %d'
        
        fig = go.Figure()

        if not default_data['empty']:
            # Target zone
            fig.add_shape(
                type='rect',
                x0=default_data['dates'][0],
                x1=default_data['dates'][-1],
                y0=target_low,
                y1=target_high,
                fillcolor=COLORS['sage'],
                opacity=0.2,
                line_width=0,
                name='Target zone',
            )

            # Min-max band
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'] + default_data['dates'][::-1],
                    y=list(default_data['max_variation']) + list(default_data['min_variation'][::-1]),
                    fill='toself',
                    fillcolor=COLORS['seafoam'],
                    opacity=0.38,
                    line_color='rgba(0,0,0,0)',
                    hoverinfo='skip',
                    name='Min-max band',
                    showlegend=True,
                )
            )

            # Average variation line
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'],
                    y=default_data['avg_variation'],
                    name='Average variation',
                    mode='lines+markers',
                    line=dict(color=COLORS['teal'], width=2.5),
                    marker=dict(size=6),
                    hovertemplate='<b>%{x|' + hover_date_fmt + '}</b><br>Avg: %{y:.2f} st<extra></extra>',
                )
            )

            # Overall average line
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'],
                    y=np.full_like(default_data['avg_variation'], default_data['overall_average']),
                    name=f'Overall average ({default_data["overall_average"]:.2f} st)',
                    mode='lines',
                    line=dict(color=COLORS['ink'], width=2, dash='dash'),
                    hovertemplate='<b>%{x|' + hover_date_fmt + '}</b><br>Overall: ' + f'{default_data["overall_average"]:.2f}' + ' st<extra></extra>',
                )
            )
        else:
            # Add empty annotation
            fig.add_annotation(
                text=default_data['message'],
                xref='paper',
                yref='paper',
                x=0.5,
                y=0.5,
                showarrow=False,
                font=dict(size=16, color='#999999'),
            )
        
        # Create time frame dropdown buttons
        timeframe_buttons = []
        for frame_name, (frame_type, start_date, end_date) in time_frames.items():
            fdata = frame_data[frame_name]
            
            # Determine hover date format for this frame
            if frame_type == 'week':
                hover_fmt = '%a, %b %d'
            else:
                hover_fmt = '%b %d, %Y'
            
            if fdata['empty']:
                # Empty state button
                timeframe_buttons.append(
                    {
                        'label': frame_name,
                        'method': 'update',
                        'args': [
                            {
                                'x': [[], [], []],
                                'y': [[], [], []],
                            },
                            {
                                'title': f'Pitch range per session over time ({frame_name})',
                                'annotations': [{
                                    'text': fdata['message'],
                                    'xref': 'paper',
                                    'yref': 'paper',
                                    'x': 0.5,
                                    'y': 0.5,
                                    'showarrow': False,
                                    'font': dict(size=16, color='#999999'),
                                }],
                                'yaxis.range': [0, fdata['upper_limit']],
                            },
                        ],
                    }
                )
            else:
                # Normal state button
                timeframe_buttons.append(
                    {
                        'label': frame_name,
                        'method': 'update',
                        'args': [
                            {
                                'x': [fdata['dates'] + fdata['dates'][::-1], fdata['dates'], fdata['dates']],
                                'y': [
                                    list(fdata['max_variation']) + list(fdata['min_variation'][::-1]),
                                    fdata['avg_variation'],
                                    np.full_like(fdata['avg_variation'], fdata['overall_average']),
                                ],
                                'hovertemplate': [
                                    'skip',
                                    '<b>%{x|' + hover_fmt + '}</b><br>Avg: %{y:.2f} st<extra></extra>',
                                    '<b>%{x|' + hover_fmt + '}</b><br>Overall: ' + f'{fdata["overall_average"]:.2f}' + ' st<extra></extra>',
                                ],
                            },
                            {
                                'title': f'Pitch range per session over time ({frame_name})',
                                'annotations': [],
                                'yaxis.range': [0, fdata['upper_limit']],
                            },
                        ],
                    }
                )

        fig.update_layout(
            title=f'Pitch range per session over time ({default_frame})',
            xaxis_title='Session date',
            yaxis_title='Pitch variation (semitones from session median)',
            template='plotly_white',
            height=600,
            hovermode='x unified',
            yaxis=dict(range=[0, default_data['upper_limit']]),
            legend=dict(
                orientation='v',
                yanchor='top',
                y=0.99,
                xanchor='left',
                x=0.01,
                font=dict(size=11),
            ),
            updatemenus=[
                {
                    'buttons': timeframe_buttons,
                    'direction': 'down',
                    'showactive': True,
                    'x': 0.0,
                    'xanchor': 'left',
                    'y': 1.12,
                    'yanchor': 'top',
                    'bgcolor': 'white',
                    'bordercolor': '#B7C7E5',
                    'borderwidth': 1,
                }
            ],
        )

        fig.show()

In [53]:
# Plot 4: Interactive WPM over time with time frame selector.
import plotly.graph_objects as go
from dateutil.relativedelta import relativedelta
from datetime import timedelta

if not records:
    print('No recordings available to plot.')
else:
    wpm_records = [record for record in records if np.isfinite(record['wpm'])]
    if not wpm_records:
        print('No WPM values available to plot.')
    else:
        latest_date = wpm_records[-1]['recording_dt'] if wpm_records else None
        
        # Calculate Monday and Sunday of this week
        monday_this_week = latest_date - timedelta(days=latest_date.weekday())
        sunday_this_week = monday_this_week + timedelta(days=6)
        
        # Calculate Monday and Sunday of last week
        monday_last_week = monday_this_week - timedelta(days=7)
        sunday_last_week = monday_last_week + timedelta(days=6)
        
        # Define time frame filters with start and end dates
        time_frames = {
            'This week': ('week', monday_this_week, sunday_this_week),
            'Last week': ('week', monday_last_week, sunday_last_week),
            'Month': ('month', latest_date - relativedelta(months=1), latest_date),
            '6 months': ('period', latest_date - relativedelta(months=6), latest_date),
            'Year': ('period', latest_date - relativedelta(years=1), latest_date),
            'All time': ('all', None, None),
        }
        
        def get_filtered_wpm_data(frame_type, start_date, end_date):
            """Filter records based on date range"""
            if frame_type == 'all':
                return wpm_records
            if start_date is None:
                return wpm_records
            return [r for r in wpm_records if start_date <= r['recording_dt'] <= end_date]
        
        # Build data for each time frame
        frame_data = {}
        target_low, target_high = 130.0, 150.0
        
        for frame_name, (frame_type, start_date, end_date) in time_frames.items():
            filtered_records = get_filtered_wpm_data(frame_type, start_date, end_date)
            
            if not filtered_records:
                # Store empty state
                frame_data[frame_name] = {
                    'empty': True,
                    'message': f"You have not done any recordings in this chosen timeframe: '{frame_name}'",
                    'y_max': 180.0,
                    'frame_type': frame_type,
                }
            else:
                dates = [r['recording_dt'] for r in filtered_records]
                wpm_values = np.array([r['wpm'] for r in filtered_records], dtype=float)
                avg_wpm = float(np.mean(wpm_values))
                
                date_numbers = mdates.date2num(dates)
                if len(wpm_values) >= 2:
                    trend_coeffs = np.polyfit(date_numbers, wpm_values, 1)
                    trend_line = np.polyval(trend_coeffs, date_numbers)
                else:
                    trend_line = wpm_values.copy()
                
                y_max = max(180.0, float(np.nanmax(wpm_values)) * 1.18 if wpm_values.size else 180.0)
                
                frame_data[frame_name] = {
                    'dates': dates,
                    'wpm_values': wpm_values,
                    'avg_wpm': avg_wpm,
                    'trend_line': trend_line,
                    'y_max': y_max,
                    'empty': False,
                    'frame_type': frame_type,
                }
        
        # Create figure with first time frame
        default_frame = 'All time'
        default_data = frame_data[default_frame]
        
        # Choose hover date format based on frame type
        hover_date_fmt = '%b %d, %Y'
        if default_data['frame_type'] == 'week':
            hover_date_fmt = '%a, %b %d'
        
        fig = go.Figure()

        if not default_data['empty']:
            # Target zone
            fig.add_shape(
                type='rect',
                x0=default_data['dates'][0],
                x1=default_data['dates'][-1],
                y0=target_low,
                y1=target_high,
                fillcolor=COLORS['sky'],
                opacity=0.2,
                line_width=0,
                name='Target zone',
            )

            # WPM line
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'],
                    y=default_data['wpm_values'],
                    name='WPM',
                    mode='lines+markers',
                    line=dict(color=COLORS['teal'], width=2.5),
                    marker=dict(size=6),
                    hovertemplate='<b>%{x|' + hover_date_fmt + '}</b><br>WPM: %{y:.1f}<extra></extra>',
                )
            )

            # Trend line
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'],
                    y=default_data['trend_line'],
                    name='Trend line',
                    mode='lines',
                    line=dict(color=COLORS['coral'], width=2.5, dash='dash'),
                    hovertemplate='<b>%{x|' + hover_date_fmt + '}</b><br>Trend: %{y:.1f}<extra></extra>',
                )
            )

            # Average line
            fig.add_trace(
                go.Scatter(
                    x=default_data['dates'],
                    y=np.full_like(default_data['wpm_values'], default_data['avg_wpm']),
                    name=f'Average WPM ({default_data["avg_wpm"]:.1f})',
                    mode='lines',
                    line=dict(color=COLORS['ink'], width=2, dash='dot'),
                    hovertemplate='<b>%{x|' + hover_date_fmt + '}</b><br>Average: ' + f'{default_data["avg_wpm"]:.1f}' + '<extra></extra>',
                )
            )
        else:
            # Add empty annotation
            fig.add_annotation(
                text=default_data['message'],
                xref='paper',
                yref='paper',
                x=0.5,
                y=0.5,
                showarrow=False,
                font=dict(size=16, color='#999999'),
            )
        
        # Create time frame dropdown buttons
        timeframe_buttons = []
        for frame_name, (frame_type, start_date, end_date) in time_frames.items():
            fdata = frame_data[frame_name]
            
            # Determine hover date format for this frame
            if frame_type == 'week':
                hover_fmt = '%a, %b %d'
            else:
                hover_fmt = '%b %d, %Y'
            
            if fdata['empty']:
                # Empty state button
                timeframe_buttons.append(
                    {
                        'label': frame_name,
                        'method': 'update',
                        'args': [
                            {
                                'x': [[], [], []],
                                'y': [[], [], []],
                            },
                            {
                                'title': f'Words per minute over time ({frame_name})',
                                'annotations': [{
                                    'text': fdata['message'],
                                    'xref': 'paper',
                                    'yref': 'paper',
                                    'x': 0.5,
                                    'y': 0.5,
                                    'showarrow': False,
                                    'font': dict(size=16, color='#999999'),
                                }],
                                'yaxis.range': [0, fdata['y_max']],
                            },
                        ],
                    }
                )
            else:
                # Normal state button
                timeframe_buttons.append(
                    {
                        'label': frame_name,
                        'method': 'update',
                        'args': [
                            {
                                'x': [fdata['dates'], fdata['dates'], fdata['dates']],
                                'y': [
                                    fdata['wpm_values'],
                                    fdata['trend_line'],
                                    np.full_like(fdata['wpm_values'], fdata['avg_wpm']),
                                ],
                                'hovertemplate': [
                                    '<b>%{x|' + hover_fmt + '}</b><br>WPM: %{y:.1f}<extra></extra>',
                                    '<b>%{x|' + hover_fmt + '}</b><br>Trend: %{y:.1f}<extra></extra>',
                                    '<b>%{x|' + hover_fmt + '}</b><br>Average: ' + f'{fdata["avg_wpm"]:.1f}' + '<extra></extra>',
                                ],
                            },
                            {
                                'title': f'Words per minute over time ({frame_name})',
                                'annotations': [],
                                'yaxis.range': [0, fdata['y_max']],
                            },
                        ],
                    }
                )

        fig.update_layout(
            title=f'Words per minute over time ({default_frame})',
            xaxis_title='Session date',
            yaxis_title='Words per minute (WPM)',
            template='plotly_white',
            height=600,
            hovermode='x unified',
            yaxis=dict(range=[0, default_data['y_max']]),
            legend=dict(
                orientation='v',
                yanchor='top',
                y=0.99,
                xanchor='left',
                x=0.01,
                font=dict(size=11),
            ),
            updatemenus=[
                {
                    'buttons': timeframe_buttons,
                    'direction': 'down',
                    'showactive': True,
                    'x': 0.0,
                    'xanchor': 'left',
                    'y': 1.12,
                    'yanchor': 'top',
                    'bgcolor': 'white',
                    'bordercolor': '#B7C7E5',
                    'borderwidth': 1,
                }
            ],
        )

        fig.show()